Null handling is one of the **most important topics** in SQL, Hive, and Spark SQL because missing values are common in real-world data. You'll use these functions constantly in ETL pipelines, reporting, and data cleansing.

---

# Null Handling Functions Comparison

| Purpose                 | Standard SQL      | Hive                     | Spark SQL       | Notes                                                    |
| ----------------------- | ----------------- | ------------------------ | --------------- | -------------------------------------------------------- |
| Check NULL              | `IS NULL`         | `IS NULL`                | `IS NULL`       | Same                                                     |
| Check NOT NULL          | `IS NOT NULL`     | `IS NOT NULL`            | `IS NOT NULL`   | Same                                                     |
| Replace NULL            | `COALESCE()`      | `coalesce()`             | `coalesce()`    | ANSI standard                                            |
| Replace NULL (2 values) | `COALESCE()`      | `nvl()`                  | `nvl()`         | `NVL` is Oracle/Hive compatible                          |
| Return NULL if Equal    | `NULLIF()`        | `nullif()` *(Hive 2.3+)* | `nullif()`      | Same behavior                                            |
| First Non-NULL          | `COALESCE()`      | `coalesce()`             | `coalesce()`    | Same                                                     |
| IF NULL                 | `CASE`            | `if()`                   | `if()`          | Hive/Spark provide `if()`                                |
| Conditional Logic       | `CASE WHEN`       | `CASE WHEN`              | `CASE WHEN`     | Same                                                     |
| Null-safe Comparison    | Database-specific | `<=>`                    | `<=>`           | Very important in Hive/Spark                             |
| Greatest                | `GREATEST()`      | `greatest()`             | `greatest()`    | Returns greatest non-null value (subject to DB behavior) |
| Least                   | `LEAST()`         | `least()`                | `least()`       | Returns least non-null value (subject to DB behavior)    |
| Count Non-NULL          | `COUNT(column)`   | `count(column)`          | `count(column)` | Ignores NULLs                                            |
| Count All Rows          | `COUNT(*)`        | `count(*)`               | `count(*)`      | Includes NULL rows                                       |

---

# Sample Data

Assume the following table:

| emp_id | name  | salary | bonus |
| ------ | ----- | ------ | ----- |
| 101    | John  | 50000  | NULL  |
| 102    | Alice | NULL   | 5000  |
| 103    | Bob   | 60000  | 2000  |
| 104    | NULL  | 45000  | NULL  |

---

# 1. IS NULL

Find employees with missing salary.

### Standard SQL

```sql
SELECT *
FROM employee
WHERE salary IS NULL;
```

### Hive

```sql
SELECT *
FROM employee
WHERE salary IS NULL;
```

### Spark SQL

```sql
SELECT *
FROM employee
WHERE salary IS NULL;
```

---

# 2. IS NOT NULL

```sql
SELECT *
FROM employee
WHERE salary IS NOT NULL;
```

---

# 3. COALESCE()

Returns the **first non-null value**.

### Example

```sql
SELECT
    name,
    coalesce(bonus,0) AS bonus
FROM employee;
```

Output

| name  | bonus |
| ----- | ----- |
| John  | 0     |
| Alice | 5000  |
| Bob   | 2000  |

---

## Multiple Columns

```sql
SELECT
coalesce(home_phone,
         office_phone,
         mobile_phone,
         'Not Available')
FROM customer;
```

Returns the first available phone number.

---

# 4. NVL()

Hive

```sql
SELECT nvl(bonus,0)
FROM employee;
```

Spark

```sql
SELECT nvl(bonus,0)
FROM employee;
```

Standard SQL

❌ No ANSI `NVL()` (Oracle supports it, but ANSI SQL recommends `COALESCE()`).

Equivalent:

```sql
SELECT coalesce(bonus,0)
FROM employee;
```

---

# 5. NULLIF()

Returns **NULL if two values are equal**.

```sql
SELECT nullif(100,100);
```

Output

```text
NULL
```

```sql
SELECT nullif(100,200);
```

Output

```text
100
```

---

# 6. IF()

Hive

```sql
SELECT if(salary IS NULL,
          'No Salary',
          'Salary Available');
```

Spark

```sql
SELECT if(salary IS NULL,
          'No Salary',
          'Salary Available');
```

Standard SQL

Uses CASE:

```sql
SELECT CASE
         WHEN salary IS NULL
           THEN 'No Salary'
         ELSE 'Salary Available'
       END;
```

---

# 7. CASE WHEN

Works in all three.

```sql
SELECT
CASE
WHEN salary IS NULL
THEN 0
ELSE salary
END
FROM employee;
```

---

# 8. NULL-safe Equality (`<=>`)

One of the most important interview questions.

Normal comparison:

```sql
SELECT NULL = NULL;
```

Output

```text
NULL
```

NULL is **unknown**, not equal to anything.

Spark/Hive:

```sql
SELECT NULL <=> NULL;
```

Output

```text
TRUE
```

Also:

```sql
SELECT
*
FROM employee
WHERE col1 <=> col2;
```

This matches rows where:

* Both columns are equal
* Both columns are NULL

This is especially useful in joins.

---

# 9. COUNT()

Suppose

| salary |
| ------ |
| 100    |
| 200    |
| NULL   |
| 300    |

```sql
SELECT count(salary)
FROM employee;
```

Output

```text
3
```

NULL is ignored.

---

```sql
SELECT count(*)
FROM employee;
```

Output

```text
4
```

All rows are counted.

---

# 10. GREATEST()

```sql
SELECT greatest(10,20,30);
```

Output

```text
30
```

---

# 11. LEAST()

```sql
SELECT least(10,20,30);
```

Output

```text
10
```

---

# Real-Time Examples

## Replace Missing Salary

```sql
SELECT
emp_id,
coalesce(salary,0)
FROM employee;
```

---

## Replace Missing Department

```sql
SELECT
coalesce(department,'Unknown')
FROM employee;
```

---

## Use Backup Columns

```sql
SELECT
coalesce(email,
         alternate_email,
         office_email)
FROM customer;
```

---

## Handle Missing Bonus

```sql
SELECT
salary + coalesce(bonus,0)
FROM employee;
```

Without `coalesce()`, if `bonus` is NULL, the entire expression becomes NULL.

---

## Join with NULL-safe Operator (Hive/Spark)

```sql
SELECT *
FROM orders o
JOIN customers c
ON o.customer_id <=> c.customer_id;
```

---

## Filter NULL Values

```sql
SELECT *
FROM employee
WHERE department IS NOT NULL;
```

---

## Calculate Average Ignoring NULL

```sql
SELECT avg(salary)
FROM employee;
```

`AVG()` automatically ignores NULL values.

---

# Aggregate Functions and NULL

| Function        | NULL Handling    |
| --------------- | ---------------- |
| `COUNT(column)` | Ignores NULL     |
| `COUNT(*)`      | Counts every row |
| `SUM()`         | Ignores NULL     |
| `AVG()`         | Ignores NULL     |
| `MIN()`         | Ignores NULL     |
| `MAX()`         | Ignores NULL     |

Example:

| Salary |
| ------ |
| 100    |
| 200    |
| NULL   |
| 300    |

```sql
SELECT
SUM(salary),
AVG(salary),
COUNT(salary)
FROM employee;
```

Output

```text
SUM   600

AVG   200

COUNT 3
```

---

# Key Differences

| Feature                  | Standard SQL                    | Hive            | Spark SQL |
| ------------------------ | ------------------------------- | --------------- | --------- |
| `COALESCE()`             | ✅                               | ✅               | ✅         |
| `NVL()`                  | ❌ (ANSI) *(Oracle supports it)* | ✅               | ✅         |
| `IF()`                   | ❌                               | ✅               | ✅         |
| `CASE WHEN`              | ✅                               | ✅               | ✅         |
| `NULLIF()`               | ✅                               | ✅ *(Hive 2.3+)* | ✅         |
| Null-safe equality `<=>` | ❌ *(not ANSI)*                  | ✅               | ✅         |
| `IS NULL`                | ✅                               | ✅               | ✅         |
| `IS NOT NULL`            | ✅                               | ✅               | ✅         |

---

# Interview Questions

### 1. Difference between `COALESCE()` and `NVL()`

| `COALESCE()`                   | `NVL()`                               |
| ------------------------------ | ------------------------------------- |
| ANSI SQL standard              | Oracle/Hive/Spark extension           |
| Accepts **multiple** arguments | Accepts **two** arguments             |
| Returns first non-NULL value   | Returns second value if first is NULL |

Example:

```sql
SELECT coalesce(NULL,NULL,100,200);
```

Output

```text
100
```

```sql
SELECT nvl(NULL,100);
```

Output

```text
100
```

---

### 2. Difference between `COUNT(*)` and `COUNT(column)`

| Function        | Result                      |
| --------------- | --------------------------- |
| `COUNT(*)`      | Counts all rows             |
| `COUNT(column)` | Counts only non-NULL values |

---

### 3. Why use `<=>` instead of `=` in Spark/Hive?

* `=` returns **NULL (unknown)** when either operand is NULL.
* `<=>` is **null-safe**:

  * `NULL <=> NULL` → `TRUE`
  * `5 <=> 5` → `TRUE`
  * `5 <=> NULL` → `FALSE`

This makes `<=>` especially useful in joins, deduplication, and change data capture (CDC) where NULLs should be treated as equal.
